# 07 — Benchmark Distribusi & Paralelisasi (Skenario 1, 2, 4, 5)

Notebook ini membuktikan secara langsung konsep-konsep utama pemrosesan terdistribusi:

| Skenario | Konsep |
|----------|--------|
| **1** | Perbandingan performa `local[*]` vs Spark Cluster |
| **2** | Partisi data per kota (data locality) |
| **4** | Scale worker horizontal |
| **5** | Fault tolerance — job tetap selesai meski satu worker mati |

> **Prasyarat:** dataset besar sudah ada di MinIO (`python -m data.generate_large_dataset`)

## Setup

In [ ]:
import os, time
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.insert(0, "/home/jovyan/work")

from analysis.spark_session import create_spark_session
from pyspark.sql import functions as F

spark = create_spark_session("07 - Benchmark")
BUCKET = "datalake"

print(f"Spark master     : {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

---
## Skenario 1 — Benchmark Performa: local[*] vs Cluster

Jalankan cell ini dua kali:
1. Saat menggunakan profile `analysis` (SPARK_MASTER=local[*])
2. Saat menggunakan profile `cluster` (SPARK_MASTER=spark://spark-master:7077)

Catat waktu masing-masing untuk perbandingan.

In [ ]:
# Load dataset
df_orders    = spark.read.csv(f"s3a://{BUCKET}/raw/large/orders/",    header=True, inferSchema=True)
df_products  = spark.read.csv(f"s3a://{BUCKET}/raw/large/products/",  header=True, inferSchema=True)
df_customers = spark.read.csv(f"s3a://{BUCKET}/raw/large/customers/", header=True, inferSchema=True)

df_orders.cache()
df_products.cache()
df_customers.cache()

# Warm-up cache
_ = df_orders.count()
_ = df_products.count()
_ = df_customers.count()

print(f"Order: {df_orders.count():,} baris")

In [ ]:
# ── Benchmark: Agregasi besar ──────────────────────────────────────────────
t0 = time.time()

df_detail = df_orders.join(df_products.select("product_id", "price", "category"), on="product_id")
df_detail = df_detail.withColumn("spend", F.col("quantity") * F.col("price"))

df_result = df_detail.groupBy("customer_id").agg(
    F.count("order_id").alias("total_orders"),
    F.sum("spend").alias("total_spend"),
    F.countDistinct("category").alias("kategori_dibeli"),
)

n = df_result.count()
elapsed = time.time() - t0

print(f"Master  : {spark.sparkContext.master}")
print(f"Hasil   : {n:,} pelanggan diproses")
print(f"Waktu   : {elapsed:.2f} detik")
print()
df_result.orderBy("total_spend", ascending=False).show(5)

In [ ]:
# Tabel perbandingan — isi manual setelah menjalankan di kedua mode
import pandas as pd
import matplotlib.pyplot as plt

hasil = [
    {"Mode": "local[*]",              "Waktu (detik)": None},  # ← isi setelah run di Tahap 2
    {"Mode": "cluster (2 worker)",    "Waktu (detik)": None},  # ← isi setelah run di Tahap 3
    {"Mode": "cluster (3 worker)",    "Waktu (detik)": None},  # ← isi setelah run di Tahap 3
]

# Isi nilai waktu di sini, contoh:
# hasil[0]["Waktu (detik)"] = 45.2
# hasil[1]["Waktu (detik)"] = 28.7
# hasil[2]["Waktu (detik)"] = 21.3

df_bench = pd.DataFrame(hasil).dropna()
if not df_bench.empty:
    df_bench["Speedup"] = (df_bench["Waktu (detik)"].iloc[0] / df_bench["Waktu (detik)"]).round(2)
    print(df_bench.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(df_bench["Mode"], df_bench["Waktu (detik)"], color=["steelblue", "darkorange", "seagreen"])
    ax.set_ylabel("Waktu (detik)")
    ax.set_title("Perbandingan Performa: local vs Cluster")
    plt.tight_layout()
    plt.show()
else:
    print("Isi nilai waktu di variabel 'hasil' setelah menjalankan benchmark di masing-masing mode.")

---
## Skenario 2 — Partisi Data per Kota (Data Locality)

Data disimpan ke MinIO dengan partisi per kota. Setiap kota menjadi folder terpisah.
Amati di Spark UI (`http://localhost:8080`) berapa task berjalan **paralel** saat query per kota.

In [ ]:
# ── Simpan dengan partisi per kota ────────────────────────────────────────
t0 = time.time()

df_with_city = df_result.join(df_customers.select("customer_id", "city"), on="customer_id")

df_with_city.write \
    .partitionBy("city") \
    .mode("overwrite") \
    .parquet(f"s3a://{BUCKET}/processed/large/customers_by_city/")

print(f"Waktu simpan partisi: {time.time() - t0:.1f} detik")
print()
print("Struktur di MinIO:")
print("  processed/large/customers_by_city/")
for city in df_customers.select("city").distinct().orderBy("city").toPandas()["city"]:
    print(f"    city={city}/")

In [ ]:
# ── Baca hanya satu kota — Spark hanya scan partisi relevan (partition pruning) ──
t0 = time.time()

df_jakarta = spark.read.parquet(f"s3a://{BUCKET}/processed/large/customers_by_city/") \
    .filter(F.col("city") == "Jakarta")

n_jakarta = df_jakarta.count()
print(f"Pelanggan Jakarta : {n_jakarta:,}")
print(f"Waktu query       : {time.time() - t0:.2f} detik")
print()
print("Bandingkan dengan query tanpa partisi (cell berikut).")

In [ ]:
# ── Query tanpa partisi — Spark scan semua data ────────────────────────────
t0 = time.time()

n_jakarta_no_part = df_with_city.filter(F.col("city") == "Jakarta").count()

print(f"Pelanggan Jakarta (tanpa partisi): {n_jakarta_no_part:,}")
print(f"Waktu query                      : {time.time() - t0:.2f} detik")

---
## Skenario 4 — Scale Worker Horizontal

Jalankan benchmark yang sama dengan jumlah worker berbeda dan catat hasilnya.

**Di terminal (luar notebook):**
```powershell
# Restart dengan N worker (ganti N = 1, 2, atau 3)
docker compose --profile cluster up -d --scale spark-worker=1

# Tunggu worker terdaftar di http://localhost:8080
# Lalu restart kernel notebook ini dan jalankan ulang cell benchmark
```

Isi tabel di bawah setelah mendapatkan waktu untuk masing-masing konfigurasi.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Isi setelah menjalankan benchmark di masing-masing konfigurasi worker ──
scale_hasil = [
    {"Worker": 1, "Waktu (detik)": None},
    {"Worker": 2, "Waktu (detik)": None},
    {"Worker": 3, "Waktu (detik)": None},
]
# Contoh:
# scale_hasil[0]["Waktu (detik)"] = 62.4
# scale_hasil[1]["Waktu (detik)"] = 38.1
# scale_hasil[2]["Waktu (detik)"] = 28.9

df_scale = pd.DataFrame(scale_hasil).dropna()
if not df_scale.empty:
    base = df_scale["Waktu (detik)"].iloc[0]
    df_scale["Speedup"]   = (base / df_scale["Waktu (detik)"]).round(2)
    df_scale["Efisiensi"] = (df_scale["Speedup"] / df_scale["Worker"]).round(2)
    print(df_scale.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df_scale["Worker"], df_scale["Waktu (detik)"], marker="o", color="steelblue")
    axes[0].set_xlabel("Jumlah Worker")
    axes[0].set_ylabel("Waktu (detik)")
    axes[0].set_title("Waktu vs Jumlah Worker")

    axes[1].plot(df_scale["Worker"], df_scale["Speedup"], marker="o", color="seagreen", label="Aktual")
    axes[1].plot(df_scale["Worker"], df_scale["Worker"].astype(float), linestyle="--", color="gray", label="Linear ideal")
    axes[1].set_xlabel("Jumlah Worker")
    axes[1].set_ylabel("Speedup")
    axes[1].set_title("Speedup vs Jumlah Worker")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Isi nilai waktu di variabel 'scale_hasil' setelah menjalankan tiap konfigurasi worker.")

---
## Skenario 5 — Fault Tolerance

Spark secara otomatis menjadwalkan ulang task yang gagal ke worker lain.
Skenario ini membuktikan bahwa job **tetap selesai** meski satu worker dimatikan di tengah eksekusi.

**Langkah:**
1. Jalankan cell berikut (job berat ~30 detik)
2. Segera setelah eksekusi dimulai, buka terminal baru dan jalankan:
   ```powershell
   docker compose --profile cluster stop spark-worker
   ```
3. Amati di Spark UI (`http://localhost:8080`) — task yang gagal di-retry di worker lain
4. Job tetap selesai (mungkin lebih lambat)

> Catatan: butuh minimal 2 worker aktif agar ada worker pengganti.

In [ ]:
# ── Job berat untuk simulasi fault tolerance ───────────────────────────────
# Jalankan cell ini, lalu segera matikan salah satu worker dari terminal

print("Job dimulai. Segera jalankan di terminal:")
print("  docker compose --profile cluster stop spark-worker")
print()

t0 = time.time()

# Join besar + agregasi bertingkat
df_fault_test = df_orders \
    .join(df_products.select("product_id", "price", "name", "category"), on="product_id") \
    .join(df_customers.select("customer_id", "city", "age"), on="customer_id") \
    .withColumn("spend", F.col("quantity") * F.col("price")) \
    .groupBy("city", "category") \
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("spend").alias("total_spend"),
        F.countDistinct("customer_id").alias("unique_customers"),
    ) \
    .orderBy("total_spend", ascending=False)

n = df_fault_test.count()
elapsed = time.time() - t0

print(f"Job selesai! ({n} kombinasi kota-kategori, {elapsed:.1f} detik)")
print()
print("Jika output di atas muncul meski worker dimatikan — Fault Tolerance BERHASIL.")
df_fault_test.show(10)

---
## Ringkasan Konsep

| Skenario | Konsep | Bukti |
|----------|--------|-------|
| 1 — Performa | Cluster lebih cepat untuk data besar | Waktu eksekusi berkurang |
| 2 — Partisi | Partition pruning mengurangi data yang di-scan | Query kota jauh lebih cepat |
| 4 — Scale | Penambahan worker meningkatkan throughput | Speedup mendekati linear |
| 5 — Fault | Task gagal di-retry otomatis di worker lain | Job selesai meski worker mati |